# LitCVit jobs on Colab

Runs `seeds.py` or `bench.py` unchanged. `/kaggle/input` is read-only on Colab, so the data and code are unpacked under `/content`, and the job is pointed at them and at a Drive folder for its output.

1. Upload the folder `cns_proj` to the top level of your Google Drive. Keep it private.
2. Runtime > Change runtime type > T4 GPU.
3. Set `JOB` below and choose Runtime > Run all. Allow Drive access when asked.
4. Keep the tab open and the laptop awake. If the session stops, run all again: finished rows are saved in Drive and aren't repeated.

Colab doesn't publish its free GPU limit. Once connected, Runtime > View resources usually estimates how long this runtime may last.

In [ ]:
JOB = "seeds"  # or "bench"

import pathlib, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
HOME = pathlib.Path("/content/drive/MyDrive/cns_proj")
assert (HOME / "litcvit-flows.zip").exists(), "upload the cns_proj folder to the top of your Drive first"

import torch
assert torch.cuda.is_available(), "no GPU: choose Runtime > Change runtime type > T4 GPU. If Colab says it can't connect to a GPU, the limit is used up for now: finished rows are safe in Drive, so run all again later"
print(torch.cuda.get_device_name(0))

In [ ]:
out = HOME / "output" / JOB
out.mkdir(parents=True, exist_ok=True)
!pip install -q timm
!unzip -q -o "{HOME}/litcvit-flows.zip" -d /content/flows
!unzip -q -o "{HOME}/litcvit-encoder.zip" -d /content/code
!python -c "import numpy, pandas, torch, timm; print('numpy', numpy.__version__, 'pandas', pandas.__version__, 'torch', torch.__version__, 'timm', timm.__version__)" | tee "{out}/versions.txt"

In [ ]:
pathlib.Path("/content/run_job.py").write_text(f"""
import importlib.util, sys
from pathlib import Path
sys.path.insert(0, "/content/code")
import light as T
T.L.DATA, T.L.OUT = Path("/content/flows"), Path("{out}")
spec = importlib.util.spec_from_file_location("job", "{HOME}/{JOB}.py")
job = importlib.util.module_from_spec(spec)
spec.loader.exec_module(job)
job.main()
""")
!cd /content && python -u run_job.py 2>&1 | tee -a "{out}/{JOB}.log"